# Transformer Translation Notebook (Colab-Friendly)

This notebook trains a TensorFlow/Keras transformer for **English -> Igbo** translation using the JSONL files in `Cleaned Data/`.

It is written specifically for a staged Colab workflow so you can test the full pipeline cheaply before moving to larger runs.

## Main Ideas

- start at `10k`
- then move to `50k`
- then `100k`
- only then try `500k`
- keep early runs at `2-3` epochs
- save checkpoints to Drive every epoch
- use early stopping with `patience=2`
- decode only a limited test sample by default

## Notebook Map

1. install and import dependencies
2. mount Drive and define the run configuration
3. locate data files and output directories
4. load JSONL records and build train/validation/test splits
5. vectorize text and build `tf.data` pipelines
6. define the transformer, loss, masks, and compile settings
7. train with checkpoints and early stopping
8. evaluate, decode examples, and save metrics back to Drive


## Recommended Workflow

1. Leave `TRAINING_STAGE = "10k"` and run the notebook once to verify the pipeline, Drive saving, and decoding path.
2. Move to `50k`, then `100k`, then `500k` only after the smaller stages look healthy.
3. Keep `OVERRIDE_EPOCHS = None` for cheap stage defaults at first.
4. Only set `OVERRIDE_EPOCHS = 5` or `10` after the model is behaving well.
5. Keep `RUN_NAME` stable if you want to resume the same run.
6. If Colab disconnects, rerun the notebook with the same `RUN_NAME` and set `RESUME_IF_AVAILABLE = True`.

## Notes On Comparisons

- The notebook truncates long sequences instead of dropping them.
- That keeps the row set more comparable across stages while still putting a cap on compute.
- Early-stage metrics are for pipeline validation and rough comparison, not final model selection.


In [1]:
%pip -q install sacrebleu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 13.2 MB/s eta 0:00:00


In [2]:
import json
import random
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

try:
    from sacrebleu.metrics import BLEU, CHRF
except ImportError:
    BLEU = None
    CHRF = None
    print("sacrebleu is not installed. BLEU and chrF will be skipped.")

# Colab GPU sessions are more stable when JIT is off and memory growth is enabled early.
tf.config.optimizer.set_jit(False)
for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

MOUNT_DRIVE = True

if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")

print("Running in Colab:", IN_COLAB)


Mounted at /content/drive
Running in Colab: True


## 1. Runtime And Experiment Setup

This section controls the stage schedule, the current run name, token limits, model size, and whether outputs should be saved to Drive.

Change `TRAINING_STAGE` when you intentionally want to move from `10k` to `50k`, `100k`, or `500k`. It does not advance automatically.


In [18]:
RANDOM_SEED = 42
keras.utils.set_random_seed(RANDOM_SEED)

# Start cheap and scale up manually. The stage does not change on its own.
STAGE_SCHEDULE = {
    "10k": {"train_sample_size": 10_000, "epochs": 2, "batch_size": 128},
    "50k": {"train_sample_size": 50_000, "epochs": 2, "batch_size": 128},
    "100k": {"train_sample_size": 100_000, "epochs": 3, "batch_size": 96},
    "500k": {"train_sample_size": 500_000, "epochs": 5, "batch_size": 64},
    "all": {"train_sample_size": None, "epochs": 8, "batch_size": 64},
}

TRAINING_STAGE = "all"
EXPERIMENT_TAG = "baseline"
# Keep this stable if you want to resume the same run from Drive.
RUN_NAME = f"transformer_{TRAINING_STAGE}_{EXPERIMENT_TAG}"

OVERRIDE_SAMPLE_SIZE = None
OVERRIDE_EPOCHS = None
OVERRIDE_BATCH_SIZE = None

# Data and vocabulary limits.
VALID_FRACTION = 0.02
SOURCE_VOCAB_SIZE = 30_000
TARGET_VOCAB_SIZE = 30_000
MAX_SOURCE_TOKENS = 40
MAX_TARGET_TOKENS = 40
TARGET_SEQUENCE_LENGTH = MAX_TARGET_TOKENS + 2

# Model size. This is intentionally modest for Colab-friendly runs.
EMBED_DIM = 256
FF_DIM = 512
NUM_HEADS = 4
NUM_ENCODER_BLOCKS = 2
NUM_DECODER_BLOCKS = 2
DROPOUT = 0.1
LEARNING_RATE = 3e-4
CLIPNORM = 1.0
EARLY_STOPPING_PATIENCE = 2
TEST_TRANSLATION_LIMIT = 1000
DECODE_BATCH_SIZE = 32
SUPPRESS_UNK_IN_DECODING = True

START_TOKEN = "[start]"
END_TOKEN = "[end]"

# Read from local /content for speed, but keep outputs in Drive.
COPY_DATA_TO_LOCAL = True
RESUME_IF_AVAILABLE = False
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_PROJECT_SUBDIR = "CS156 Final Assignment"
DRIVE_OUTPUT_SUBDIR = "Transformer Outputs"

# Resolve the final stage settings after any manual overrides.
stage_config = STAGE_SCHEDULE[TRAINING_STAGE]
TRAIN_SAMPLE_SIZE = stage_config["train_sample_size"] if OVERRIDE_SAMPLE_SIZE is None else OVERRIDE_SAMPLE_SIZE
EPOCHS = stage_config["epochs"] if OVERRIDE_EPOCHS is None else OVERRIDE_EPOCHS
BATCH_SIZE = stage_config["batch_size"] if OVERRIDE_BATCH_SIZE is None else OVERRIDE_BATCH_SIZE
CACHE_DATASETS = TRAIN_SAMPLE_SIZE is not None and TRAIN_SAMPLE_SIZE <= 100_000

print("Training stage:", TRAINING_STAGE)
print("Run name:", RUN_NAME)
print("Train sample size:", "all" if TRAIN_SAMPLE_SIZE is None else TRAIN_SAMPLE_SIZE)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Cache datasets:", CACHE_DATASETS)
print("Decode batch size:", DECODE_BATCH_SIZE)

Training stage: all
Run name: transformer_all_baseline
Train sample size: all
Epochs: 8
Batch size: 64
Cache datasets: False
Decode batch size: 32


In [5]:
# Prefer the explicit Drive project folder first, then fall back to the current working directory.
DRIVE_ROOT = Path("/content/drive/MyDrive") if IN_COLAB and MOUNT_DRIVE else None
DRIVE_PROJECT_DIR = DRIVE_ROOT / DRIVE_PROJECT_SUBDIR if DRIVE_ROOT is not None else None
PROJECT_DIR_CANDIDATES = []

if DRIVE_PROJECT_DIR is not None:
    PROJECT_DIR_CANDIDATES.append(DRIVE_PROJECT_DIR)

PROJECT_DIR_CANDIDATES.extend([Path.cwd(), Path.cwd().parent])


def resolve_project_dir(candidates):
    for candidate in candidates:
        if (candidate / "Cleaned Data" / "Combined_train.jsonl").exists():
            return candidate
    return None


PROJECT_DIR = resolve_project_dir(PROJECT_DIR_CANDIDATES)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not locate the project directory. Put the project in Google Drive or update DRIVE_PROJECT_SUBDIR."
    )

# In Colab, outputs should land in Drive so checkpoints survive disconnects.
if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    if DRIVE_PROJECT_DIR is None:
        raise FileNotFoundError("Google Drive is not mounted, so outputs cannot be saved there.")
    OUTPUT_DIR = DRIVE_PROJECT_DIR / DRIVE_OUTPUT_SUBDIR
else:
    OUTPUT_DIR = PROJECT_DIR / DRIVE_OUTPUT_SUBDIR

DRIVE_DATA_DIR = PROJECT_DIR / "Cleaned Data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

drive_train_path = DRIVE_DATA_DIR / "Combined_train.jsonl"
drive_test_path = DRIVE_DATA_DIR / "Combined_test.jsonl"

# Copy large JSONL files to local disk for faster reads during training.
if COPY_DATA_TO_LOCAL and IN_COLAB:
    LOCAL_DATA_DIR = Path("/content/cs156_transformer_data")
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for source_path in [drive_train_path, drive_test_path]:
        local_path = LOCAL_DATA_DIR / source_path.name
        if not local_path.exists() or local_path.stat().st_size != source_path.stat().st_size:
            shutil.copy2(source_path, local_path)
    TRAIN_PATH = LOCAL_DATA_DIR / drive_train_path.name
    TEST_PATH = LOCAL_DATA_DIR / drive_test_path.name
else:
    TRAIN_PATH = drive_train_path
    TEST_PATH = drive_test_path

BEST_MODEL_PATH = OUTPUT_DIR / f"{RUN_NAME}_best.keras"
FINAL_MODEL_PATH = OUTPUT_DIR / f"{RUN_NAME}_final.keras"
EPOCH_MODEL_TEMPLATE = str(OUTPUT_DIR / f"{RUN_NAME}_epoch{{epoch:02d}}.keras")
HISTORY_PATH = OUTPUT_DIR / f"{RUN_NAME}_history.csv"
METRICS_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_metrics.csv"
PREVIEW_PATH = OUTPUT_DIR / f"{RUN_NAME}_test_preview.csv"
SOURCE_VOCAB_PATH = OUTPUT_DIR / f"{RUN_NAME}_source_vocab.txt"
TARGET_VOCAB_PATH = OUTPUT_DIR / f"{RUN_NAME}_target_vocab.txt"
RUN_CONFIG_PATH = OUTPUT_DIR / f"{RUN_NAME}_config.json"

# Save the resolved run configuration next to the checkpoints for traceability.
run_config = {
    "run_name": RUN_NAME,
    "training_stage": TRAINING_STAGE,
    "train_sample_size": TRAIN_SAMPLE_SIZE,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "max_source_tokens": MAX_SOURCE_TOKENS,
    "max_target_tokens": MAX_TARGET_TOKENS,
    "embed_dim": EMBED_DIM,
    "ff_dim": FF_DIM,
    "num_heads": NUM_HEADS,
    "num_encoder_blocks": NUM_ENCODER_BLOCKS,
    "num_decoder_blocks": NUM_DECODER_BLOCKS,
    "dropout": DROPOUT,
    "learning_rate": LEARNING_RATE,
    "clipnorm": CLIPNORM,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "test_translation_limit": TEST_TRANSLATION_LIMIT,
    "copy_data_to_local": COPY_DATA_TO_LOCAL,
    "resume_if_available": RESUME_IF_AVAILABLE,
    "save_outputs_to_drive": SAVE_OUTPUTS_TO_DRIVE,
    "project_dir": str(PROJECT_DIR),
    "train_path": str(TRAIN_PATH),
    "test_path": str(TEST_PATH),
    "output_dir": str(OUTPUT_DIR),
    "best_model_path": str(BEST_MODEL_PATH),
    "final_model_path": str(FINAL_MODEL_PATH),
}
RUN_CONFIG_PATH.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

print("Project directory:", PROJECT_DIR)
print("Training file:", TRAIN_PATH)
print("Test file:", TEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Run config path:", RUN_CONFIG_PATH)
print("Best checkpoint path:", BEST_MODEL_PATH)


Project directory: /content/drive/MyDrive/CS156 Final Assignment
Training file: /content/cs156_transformer_data/Combined_train.jsonl
Test file: /content/cs156_transformer_data/Combined_test.jsonl
Output directory: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs
Run config path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_config.json
Best checkpoint path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_best.keras


## 2. Data Preparation

The next cells normalize text, tokenize it, truncate long examples, optionally subsample the training pool for cheap stages, and then create the train/validation/test splits used by the notebook.


In [6]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def normalize_text(text):
    text = text.lower().strip()
    text = text.replace("\u2019", "'")
    text = text.replace("\u2018", "'")
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = text.replace("\u02bc", "'")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def tokenize(text):
    return TOKEN_PATTERN.findall(normalize_text(text))


def build_record(item):
    source_tokens = tokenize(item["English"])
    target_tokens = tokenize(item["Igbo"])

    if not source_tokens or not target_tokens:
        return None

    # Truncate rather than drop long rows so stage-to-stage comparisons stay closer.
    was_truncated = len(source_tokens) > MAX_SOURCE_TOKENS or len(target_tokens) > MAX_TARGET_TOKENS
    source_tokens = source_tokens[:MAX_SOURCE_TOKENS]
    target_tokens = target_tokens[:MAX_TARGET_TOKENS]

    source_text = " ".join(source_tokens)
    target_core = " ".join(target_tokens)
    target_text = f"{START_TOKEN} {target_core} {END_TOKEN}"

    return {
        "source_text": source_text,
        "target_text": target_text,
        "reference_text": target_core,
        "source_length": len(source_tokens),
        "target_length": len(target_tokens),
        "was_truncated": was_truncated,
    }


def load_jsonl_records(path, sample_size=None, seed=42):
    rng = random.Random(seed)
    records = []
    total_valid = 0
    truncated = 0

    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue

            item = json.loads(line)
            record = build_record(item)
            if record is None:
                continue

            total_valid += 1
            truncated += int(record["was_truncated"])

            if sample_size is None:
                records.append(record)
                continue

            if len(records) < sample_size:
                records.append(record)
                continue

            # Reservoir sampling lets cheap stages read a huge file without keeping every row.
            replacement_index = rng.randint(0, total_valid - 1)
            if replacement_index < sample_size:
                records[replacement_index] = record

    stats = {
        "total_valid": total_valid,
        "truncated": truncated,
        "truncated_pct": round((truncated / total_valid) * 100, 2) if total_valid else 0.0,
    }
    return records, stats


def split_train_validation(records, val_fraction=0.02, seed=42):
    shuffled = records.copy()
    random.Random(seed).shuffle(shuffled)
    val_size = max(1000, int(len(shuffled) * val_fraction))
    val_records = shuffled[:val_size]
    train_records = shuffled[val_size:]
    return train_records, val_records


def summarize_split(name, records):
    return {
        "split": name,
        "rows": len(records),
        "avg_source_tokens": round(sum(row["source_length"] for row in records) / len(records), 2),
        "avg_target_tokens": round(sum(row["target_length"] for row in records) / len(records), 2),
        "truncated_pct": round(sum(int(row["was_truncated"]) for row in records) / len(records) * 100, 2),
    }


In [7]:
sample_size_label = "all" if TRAIN_SAMPLE_SIZE is None else TRAIN_SAMPLE_SIZE

train_pool_records, train_pool_stats = load_jsonl_records(TRAIN_PATH, TRAIN_SAMPLE_SIZE, RANDOM_SEED)
test_records, test_stats = load_jsonl_records(TEST_PATH, None, RANDOM_SEED)
train_records, val_records = split_train_validation(train_pool_records, VALID_FRACTION, RANDOM_SEED)

print("Available valid train rows in file:", train_pool_stats["total_valid"])
print("Stage sample before split:", sample_size_label if TRAIN_SAMPLE_SIZE is None else len(train_pool_records))
print("Train pool truncated pct:", train_pool_stats["truncated_pct"])
print("Test truncated pct:", test_stats["truncated_pct"])

summary_df = pd.DataFrame(
    [
        summarize_split("train", train_records),
        summarize_split("validation", val_records),
        summarize_split("test", test_records),
    ]
)

display(summary_df)
display(pd.DataFrame(train_records[:5]))


Available valid train rows in file: 534064
Stage sample before split: all
Train pool truncated pct: 13.82
Test truncated pct: 25.82


,split,rows,avg_source_tokens,avg_target_tokens,truncated_pct
0,train,523383,18.21,22.66,13.82
1,validation,10681,18.23,22.64,14.01
2,test,6390,24.25,27.04,25.82


,source_text,target_text,reference_text,source_length,target_length,was_truncated
0,one of her clients was a woman named aksamai s...,[start] otu n ' ime ndị ọ na - akwara ákwà bụ ...,otu n ' ime ndị ọ na - akwara ákwà bụ otu nwaa...,25,38,False
1,what helped those loyal servants of god to sta...,[start] gịnị nyeere ndị a kwesịrị ntụkwasị obi...,"gịnị nyeere ndị a kwesịrị ntụkwasị obi , bú ̣ ...",11,18,False
2,food nutritionists are saying that beans is a ...,[start] ndị ọkachamara n ' ihe gbasara nri na ...,ndị ọkachamara n ' ihe gbasara nri na ihe ha n...,19,38,False
3,the geneva bible translators gave particular a...,[start] ndị sụgharịrị geneva bible lekwasịrị a...,ndị sụgharịrị geneva bible lekwasịrị anya kpọm...,18,18,False
4,"it was very good . ""","[start] ha dị nnọọ mma . "" [end]","ha dị nnọọ mma . """,6,6,False


In [19]:
from collections import Counter


def build_vocabulary(texts, max_tokens):
    counter = Counter()
    for text in texts:
        counter.update(text.split())

    vocab = ["", "[UNK]"]
    vocab.extend(token for token, _ in counter.most_common(max_tokens - len(vocab)))
    return vocab


# Build vocabularies in plain Python so Colab never has to adapt preprocessing layers on the GPU.
source_vocab = build_vocabulary((row["source_text"] for row in train_records), SOURCE_VOCAB_SIZE)
target_vocab = build_vocabulary((row["target_text"] for row in train_records), TARGET_VOCAB_SIZE)

source_token_lookup = {token: index for index, token in enumerate(source_vocab)}
target_token_lookup = {token: index for index, token in enumerate(target_vocab)}
target_index_lookup = dict(enumerate(target_vocab))

SOURCE_PAD_ID = 0
TARGET_PAD_ID = 0
SOURCE_UNK_ID = source_token_lookup["[UNK]"]
TARGET_UNK_ID = target_token_lookup.get("[UNK]", None)

if START_TOKEN not in target_token_lookup or END_TOKEN not in target_token_lookup:
    raise ValueError("[start] or [end] token is missing from the target vocabulary.")

START_TOKEN_ID = target_token_lookup[START_TOKEN]
END_TOKEN_ID = target_token_lookup[END_TOKEN]


def encode_source_text(text):
    token_ids = [source_token_lookup.get(token, SOURCE_UNK_ID) for token in text.split()]
    token_ids = token_ids[:MAX_SOURCE_TOKENS]
    if len(token_ids) < MAX_SOURCE_TOKENS:
        token_ids += [SOURCE_PAD_ID] * (MAX_SOURCE_TOKENS - len(token_ids))
    return token_ids


def encode_target_text(text):
    token_ids = [target_token_lookup.get(token, TARGET_UNK_ID) for token in text.split()]
    token_ids = token_ids[:TARGET_SEQUENCE_LENGTH]
    if len(token_ids) < TARGET_SEQUENCE_LENGTH:
        token_ids += [TARGET_PAD_ID] * (TARGET_SEQUENCE_LENGTH - len(token_ids))
    return token_ids

SOURCE_VOCAB_PATH.write_text("\n".join(source_vocab), encoding="utf-8")
TARGET_VOCAB_PATH.write_text("\n".join(target_vocab), encoding="utf-8")

print("Source vocabulary size:", len(source_vocab))
print("Target vocabulary size:", len(target_vocab))
print("Start token id:", START_TOKEN_ID)
print("End token id:", END_TOKEN_ID)
print("Source vocab path:", SOURCE_VOCAB_PATH)
print("Target vocab path:", TARGET_VOCAB_PATH)

Source vocabulary size: 30000
Target vocabulary size: 30000
Start token id: 3
End token id: 4
Source vocab path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_source_vocab.txt
Target vocab path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_target_vocab.txt


In [9]:
AUTOTUNE = tf.data.AUTOTUNE


def encode_records(records):
    encoder_inputs = np.asarray([encode_source_text(row["source_text"]) for row in records], dtype=np.int32)
    target_tokens = np.asarray([encode_target_text(row["target_text"]) for row in records], dtype=np.int32)
    decoder_inputs = target_tokens[:, :-1]
    decoder_targets = target_tokens[:, 1:]
    return encoder_inputs, decoder_inputs, decoder_targets


def make_dataset(records, training=False):
    encoder_inputs, decoder_inputs, decoder_targets = encode_records(records)

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            {"encoder_inputs": encoder_inputs, "decoder_inputs": decoder_inputs},
            decoder_targets,
        )
    )
    if training:
        shuffle_buffer = min(len(records), 20_000)
        dataset = dataset.shuffle(shuffle_buffer, seed=RANDOM_SEED, reshuffle_each_iteration=False)

    if CACHE_DATASETS:
        dataset = dataset.cache()

    dataset = dataset.batch(BATCH_SIZE)
    return dataset.prefetch(AUTOTUNE)


train_ds = make_dataset(train_records, training=True)
val_ds = make_dataset(val_records)
test_ds = make_dataset(test_records)

print("Train batches:", len(train_ds))
print("Validation batches:", len(val_ds))
print("Test batches:", len(test_ds))

Train batches: 8178
Validation batches: 167
Test batches: 100


## 3. Transformer Definition

This section defines the masked loss, padding-mask helper, positional embeddings, encoder, decoder, and the compiled Keras model.

The attention blocks use **explicit padding masks** rather than relying on automatic Keras masking. That makes the masking path easier to reason about and more stable in Colab.


In [10]:
@keras.utils.register_keras_serializable(package="cs156")
def masked_loss(y_true, y_pred):
    # Ignore padded target positions when computing token loss.
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    mask = tf.cast(tf.not_equal(y_true, 0), loss.dtype)
    loss *= mask
    return tf.reduce_sum(loss) / tf.maximum(tf.reduce_sum(mask), 1.0)


@keras.utils.register_keras_serializable(package="cs156")
def masked_accuracy(y_true, y_pred):
    # Accuracy should measure real tokens only, not padding.
    y_pred = tf.argmax(y_pred, axis=-1, output_type=y_true.dtype)
    matches = tf.cast(tf.equal(y_true, y_pred), tf.float32)
    mask = tf.cast(tf.not_equal(y_true, 0), tf.float32)
    matches *= mask
    return tf.reduce_sum(matches) / tf.maximum(tf.reduce_sum(mask), 1.0)


@keras.utils.register_keras_serializable(package="cs156")
class PaddingMask(layers.Layer):
    # Create an explicit boolean padding mask from token ids.
    def call(self, inputs):
        return tf.not_equal(inputs, 0)

    def get_config(self):
        return super().get_config()


@keras.utils.register_keras_serializable(package="cs156")
class PositionalEmbedding(layers.Layer):
    # Combine learned token embeddings with learned position embeddings.
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.token_embeddings = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
        )
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length,
            output_dim=embed_dim,
        )

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_tokens = self.token_embeddings(inputs)
        embedded_positions = self.position_embeddings(positions)
        return embedded_tokens + embedded_positions

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config


@keras.utils.register_keras_serializable(package="cs156")
class TransformerEncoder(layers.Layer):
    # Standard encoder block: self-attention, feed-forward, residuals, and layer norms.
    def __init__(self, embed_dim, dense_dim, num_heads, dropout, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout,
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)

    def call(self, inputs, training=False, padding_mask=None):
        attention_mask = None
        # Pass an explicit mask to attention rather than relying on automatic Keras masks.
        if padding_mask is not None:
            attention_mask = tf.cast(padding_mask[:, tf.newaxis, :], dtype="bool")

        attention_output = self.attention(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=attention_mask,
            training=training,
        )
        attention_output = self.dropout_1(attention_output, training=training)
        proj_input = self.layernorm_1(inputs + attention_output)
        proj_output = self.dense_proj(proj_input)
        proj_output = self.dropout_2(proj_output, training=training)
        return self.layernorm_2(proj_input + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
                "dropout": self.dropout,
            }
        )
        return config


@keras.utils.register_keras_serializable(package="cs156")
class TransformerDecoder(layers.Layer):
    # Decoder block: causal self-attention, cross-attention, feed-forward, residuals.
    def __init__(self, embed_dim, dense_dim, num_heads, dropout, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.dropout = dropout
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout,
        )
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=dropout,
        )
        self.dense_proj = keras.Sequential(
            [
                layers.Dense(dense_dim, activation="relu"),
                layers.Dense(embed_dim),
            ]
        )
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.dropout_1 = layers.Dropout(dropout)
        self.dropout_2 = layers.Dropout(dropout)
        self.dropout_3 = layers.Dropout(dropout)

    def get_causal_attention_mask(self, inputs):
        # Prevent the decoder from attending to future target positions.
        batch_size = tf.shape(inputs)[0]
        sequence_length = tf.shape(inputs)[1]
        mask = tf.linalg.band_part(
            tf.ones((sequence_length, sequence_length), dtype="bool"),
            -1,
            0,
        )
        mask = tf.expand_dims(mask, axis=0)
        return tf.tile(mask, [batch_size, 1, 1])

    def call(self, inputs, encoder_outputs, training=False, decoder_padding_mask=None, encoder_padding_mask=None):
        causal_mask = self.get_causal_attention_mask(inputs)
        self_attention_mask = causal_mask

        # Combine the causal mask with the decoder padding mask.
        if decoder_padding_mask is not None:
            padding_mask = tf.cast(decoder_padding_mask[:, tf.newaxis, :], dtype="bool")
            self_attention_mask = tf.logical_and(causal_mask, padding_mask)

        attention_output_1 = self.attention_1(
            query=inputs,
            value=inputs,
            key=inputs,
            attention_mask=self_attention_mask,
            training=training,
        )
        attention_output_1 = self.dropout_1(attention_output_1, training=training)
        out_1 = self.layernorm_1(inputs + attention_output_1)

        cross_attention_mask = None
        # Reuse the encoder padding mask during cross-attention.
        if encoder_padding_mask is not None:
            cross_attention_mask = tf.cast(encoder_padding_mask[:, tf.newaxis, :], dtype="bool")

        attention_output_2 = self.attention_2(
            query=out_1,
            value=encoder_outputs,
            key=encoder_outputs,
            attention_mask=cross_attention_mask,
            training=training,
        )
        attention_output_2 = self.dropout_2(attention_output_2, training=training)
        out_2 = self.layernorm_2(out_1 + attention_output_2)

        proj_output = self.dense_proj(out_2)
        proj_output = self.dropout_3(proj_output, training=training)
        return self.layernorm_3(out_2 + proj_output)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "embed_dim": self.embed_dim,
                "dense_dim": self.dense_dim,
                "num_heads": self.num_heads,
                "dropout": self.dropout,
            }
        )
        return config


CUSTOM_OBJECTS = {
    "masked_loss": masked_loss,
    "masked_accuracy": masked_accuracy,
    "PaddingMask": PaddingMask,
    "PositionalEmbedding": PositionalEmbedding,
    "TransformerEncoder": TransformerEncoder,
    "TransformerDecoder": TransformerDecoder,
}

In [11]:
keras.backend.clear_session()


def build_transformer_model():
    encoder_inputs = keras.Input(shape=(MAX_SOURCE_TOKENS,), dtype="int32", name="encoder_inputs")
    decoder_inputs = keras.Input(shape=(TARGET_SEQUENCE_LENGTH - 1,), dtype="int32", name="decoder_inputs")

    # Build explicit padding masks from the raw token ids.
    encoder_mask = PaddingMask(name="encoder_padding_mask")(encoder_inputs)
    decoder_mask = PaddingMask(name="decoder_padding_mask")(decoder_inputs)

    x = PositionalEmbedding(
        sequence_length=MAX_SOURCE_TOKENS,
        vocab_size=len(source_vocab),
        embed_dim=EMBED_DIM,
        name="encoder_positional_embedding",
    )(encoder_inputs)

    for block_index in range(NUM_ENCODER_BLOCKS):
        x = TransformerEncoder(
            embed_dim=EMBED_DIM,
            dense_dim=FF_DIM,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
            name=f"encoder_block_{block_index + 1}",
        )(x, padding_mask=encoder_mask)

    encoder_outputs = x

    x = PositionalEmbedding(
        sequence_length=TARGET_SEQUENCE_LENGTH - 1,
        vocab_size=len(target_vocab),
        embed_dim=EMBED_DIM,
        name="decoder_positional_embedding",
    )(decoder_inputs)

    for block_index in range(NUM_DECODER_BLOCKS):
        x = TransformerDecoder(
            embed_dim=EMBED_DIM,
            dense_dim=FF_DIM,
            num_heads=NUM_HEADS,
            dropout=DROPOUT,
            name=f"decoder_block_{block_index + 1}",
        )(x, encoder_outputs, decoder_padding_mask=decoder_mask, encoder_padding_mask=encoder_mask)

    x = layers.Dropout(DROPOUT)(x)
    decoder_outputs = layers.Dense(len(target_vocab), name="token_logits")(x)

    model = keras.Model(
        inputs={"encoder_inputs": encoder_inputs, "decoder_inputs": decoder_inputs},
        outputs=decoder_outputs,
        name="english_to_igbo_transformer",
    )

    # Adam with clipnorm is a practical default for transformer training in Colab.
    optimizer = keras.optimizers.Adam(
        learning_rate=LEARNING_RATE,
        beta_1=0.9,
        beta_2=0.98,
        epsilon=1e-9,
        clipnorm=CLIPNORM,
    )

    model.compile(
        optimizer=optimizer,
        loss=masked_loss,
        metrics=[masked_accuracy],
    )
    return model


# Resume from the best saved checkpoint if requested; otherwise build a fresh model.
if RESUME_IF_AVAILABLE and BEST_MODEL_PATH.exists():
    transformer = keras.models.load_model(BEST_MODEL_PATH, custom_objects=CUSTOM_OBJECTS)
    print("Resumed from:", BEST_MODEL_PATH)
else:
    transformer = build_transformer_model()

transformer.summary()

Model: "english_to_igbo_transformer"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_positional… │ (None, 40, 256)   │  7,690,240 │ encoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_padding_ma… │ (None, 40)        │          0 │ encoder_inputs[0… │
│ (PaddingMask)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, 41)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_block_1     │ (None, 40, 256)   │    527,104 │ encoder_position… │
│ (TransformerEncode… │                   │            │ encoder_padding_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_positional… │ (None, 41, 256)   │  7,690,496 │ decoder_inputs[0… │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_block_2     │ (None, 40, 256)   │    527,104 │ encoder_block_1[… │
│ (TransformerEncode… │                   │            │ encoder_padding_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_padding_ma… │ (None, 41)        │          0 │ decoder_inputs[0… │
│ (PaddingMask)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_block_1     │ (None, 41, 256)   │    790,784 │ decoder_position… │
│ (TransformerDecode… │                   │            │ encoder_block_2[… │
│                     │                   │            │ decoder_padding_… │
│                     │                   │            │ encoder_padding_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_block_2     │ (None, 41, 256)   │    790,784 │ decoder_block_1[… │
│ (TransformerDecode… │                   │            │ encoder_block_2[… │
│                     │                   │            │ decoder_padding_… │
│                     │                   │            │ encoder_padding_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_16          │ (None, 41, 256)   │          0 │ decoder_block_2[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_logits        │ (None, 41, 30000) │  7,710,000 │ dropout_16[0][0]  │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 25,726,512 (98.14 MB)

 Trainable params: 25,726,512 (98.14 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# Save every epoch, keep a best-validation checkpoint, and stop early when progress stalls.
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=EPOCH_MODEL_TEMPLATE,
        monitor="val_loss",
        save_best_only=False,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(BEST_MODEL_PATH),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=EARLY_STOPPING_PATIENCE,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(
        filename=str(HISTORY_PATH),
        append=RESUME_IF_AVAILABLE and HISTORY_PATH.exists(),
    ),
    keras.callbacks.TerminateOnNaN(),
]

print("Epoch checkpoint template:", EPOCH_MODEL_TEMPLATE)
print("Best checkpoint path:", BEST_MODEL_PATH)
print("History path:", HISTORY_PATH)


Epoch checkpoint template: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_epoch{epoch:02d}.keras
Best checkpoint path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_best.keras
History path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_history.csv


## 4. Training, Saving, And Evaluation

Training writes a checkpoint every epoch, keeps a separate best-validation checkpoint, saves a final model artifact, and then runs both loss-based evaluation and a small decoding pass for qualitative inspection.


In [13]:
# Fit the model on the staged training split.
history = transformer.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# Save a final artifact even if early stopping already restored the best weights.
transformer.save(FINAL_MODEL_PATH)
print("Final model path:", FINAL_MODEL_PATH)

history_df = pd.DataFrame(history.history)
display(history_df)


Epoch 1/8
8178/8178 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 3.7603 - masked_accuracy: 0.3690
Epoch 1: saving model to /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_epoch01.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_epoch01.keras

Epoch 1: val_loss improved from None to 2.36369, saving model to /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_best.keras
8178/8178 ━━━━━━━━━━━━━━━━━━━━ 555s 57ms/step - loss: 3.1069 - masked_accuracy: 0.4468 - val_loss: 2.3637 - val_masked_accuracy: 0.5493
Epoch 2/8
8177/8178 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 2.4346 - masked_accuracy: 0.5417
Epoch 2: saving model to /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transfor

,loss,masked_accuracy,val_loss,val_masked_accuracy
0,3.106896,0.446801,2.363689,0.549335
1,2.346338,0.554663,2.064375,0.596049
2,2.112595,0.590541,1.934865,0.618320
3,1.989937,0.610916,1.873582,0.631655
4,1.912915,0.624469,1.832231,0.639670
5,1.859062,0.634446,1.807664,0.644608
6,1.818744,0.641975,1.792116,0.648822
7,1.788108,0.647906,1.778136,0.651729


In [14]:
# Prefer the best checkpoint for evaluation, then fall back to the final artifact.
if BEST_MODEL_PATH.exists():
    model_to_evaluate = keras.models.load_model(BEST_MODEL_PATH, custom_objects=CUSTOM_OBJECTS)
    chosen_model_path = BEST_MODEL_PATH
elif FINAL_MODEL_PATH.exists():
    model_to_evaluate = keras.models.load_model(FINAL_MODEL_PATH, custom_objects=CUSTOM_OBJECTS)
    chosen_model_path = FINAL_MODEL_PATH
else:
    model_to_evaluate = transformer
    chosen_model_path = None

print("Evaluation model:", chosen_model_path if chosen_model_path is not None else "in-memory model")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'encoder_positional_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'encoder_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'decoder_positional_e

Evaluation model: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_best.keras


In [15]:
# Report both validation and test set losses for the saved model.
val_results = model_to_evaluate.evaluate(val_ds, return_dict=True, verbose=1)
test_results = model_to_evaluate.evaluate(test_ds, return_dict=True, verbose=1)

loss_df = pd.DataFrame(
    [
        {"split": "validation", **val_results},
        {"split": "test", **test_results},
    ]
)

display(loss_df)


167/167 ━━━━━━━━━━━━━━━━━━━━ 13s 37ms/step - loss: 1.7781 - masked_accuracy: 0.6517
100/100 ━━━━━━━━━━━━━━━━━━━━ 16s 166ms/step - loss: 2.6645 - masked_accuracy: 0.5417


,split,loss,masked_accuracy
0,validation,1.778136,0.651729
1,test,2.664520,0.541720


In [20]:
def greedy_decode_batch(model, texts, max_decoded_tokens=MAX_TARGET_TOKENS, batch_size=DECODE_BATCH_SIZE):
    # Batch decoding reduces Python overhead and runs the model on many examples at once.
    normalized_texts = [" ".join(tokenize(text)[:MAX_SOURCE_TOKENS]) for text in texts]
    all_predictions = []

    for start in range(0, len(normalized_texts), batch_size):
        batch_texts = normalized_texts[start:start + batch_size]
        encoder_inputs = tf.convert_to_tensor(
            [encode_source_text(text) for text in batch_texts],
            dtype=tf.int32,
        )
        current_batch_size = len(batch_texts)

        decoder_inputs = np.zeros((current_batch_size, TARGET_SEQUENCE_LENGTH - 1), dtype="int32")
        decoder_inputs[:, 0] = START_TOKEN_ID

        generated_token_ids = [[] for _ in range(current_batch_size)]
        finished = np.zeros(current_batch_size, dtype=bool)

        for step in range(max_decoded_tokens):
            decoder_inputs_tensor = tf.convert_to_tensor(decoder_inputs, dtype=tf.int32)
            predictions = model(
                {
                    "encoder_inputs": encoder_inputs,
                    "decoder_inputs": decoder_inputs_tensor,
                },
                training=False,
            )

            step_logits = predictions[:, step, :]

            if SUPPRESS_UNK_IN_DECODING and TARGET_UNK_ID is not None:
                blocked_ids = tf.constant([0, START_TOKEN_ID, TARGET_UNK_ID], dtype=tf.int32)
                blocked_mask = tf.reduce_sum(
                    tf.one_hot(blocked_ids, depth=tf.shape(step_logits)[-1], dtype=step_logits.dtype),
                    axis=0,
                )
                step_logits = step_logits - blocked_mask[tf.newaxis, :] * tf.cast(1e9, step_logits.dtype)

            next_token_ids = tf.argmax(step_logits, axis=-1, output_type=tf.int32).numpy()

            for row_index, token_id in enumerate(next_token_ids):
                if finished[row_index]:
                    continue

                token_id = int(token_id)
                if token_id in {0, END_TOKEN_ID}:
                    finished[row_index] = True
                    continue

                generated_token_ids[row_index].append(token_id)

                if step + 1 < decoder_inputs.shape[1]:
                    decoder_inputs[row_index, step + 1] = token_id

            if finished.all():
                break

        for token_ids in generated_token_ids:
            tokens = [target_index_lookup.get(token_id, "") for token_id in token_ids]
            tokens = [token for token in tokens if token and token not in {START_TOKEN, END_TOKEN}]
            all_predictions.append(" ".join(tokens))

    return all_predictions


def decode_sequence(model, text, max_decoded_tokens=MAX_TARGET_TOKENS):
    # Reuse the batched decoder so single-sentence and multi-sentence decoding stay consistent.
    return greedy_decode_batch(model, [text], max_decoded_tokens=max_decoded_tokens, batch_size=1)[0]


def evaluate_translations(model, records, sample_limit=200):
    # Decode only a limited sample by default to avoid spending too much time on evaluation.
    rows = records if sample_limit is None else records[:sample_limit]
    source_texts = [row["source_text"] for row in rows]
    reference_texts = [row["reference_text"] for row in rows]

    predictions = greedy_decode_batch(
        model,
        source_texts,
        max_decoded_tokens=MAX_TARGET_TOKENS,
        batch_size=DECODE_BATCH_SIZE,
    )
    references = [[reference_text] for reference_text in reference_texts]
    preview_rows = []

    for row, predicted_text, reference_text in zip(rows, predictions, reference_texts):
        preview_rows.append(
            {
                "English": row["source_text"],
                "Reference Igbo": reference_text,
                "Predicted Igbo": predicted_text,
            }
        )

    exact_match = sum(pred == refs[0] for pred, refs in zip(predictions, references)) / len(predictions)
    bleu_score = BLEU().corpus_score(predictions, references).score if BLEU is not None else float("nan")
    chrf_score = CHRF().corpus_score(predictions, references).score if CHRF is not None else float("nan")

    metrics_df = pd.DataFrame(
        {
            "metric": ["Exact match", "BLEU", "chrF"],
            "value": [round(exact_match, 4), round(bleu_score, 2), round(chrf_score, 2)],
        }
    )
    preview_df = pd.DataFrame(preview_rows)
    return metrics_df, preview_df


metrics_df, preview_df = evaluate_translations(
    model_to_evaluate,
    test_records,
    sample_limit=TEST_TRANSLATION_LIMIT,
)

metrics_df.to_csv(METRICS_PATH, index=False)
preview_df.to_csv(PREVIEW_PATH, index=False)

display(metrics_df)
display(preview_df.head(20))

print("Metrics path:", METRICS_PATH)
print("Preview path:", PREVIEW_PATH)

,metric,value
0,Exact match,0.015
1,BLEU,39.680
2,chrF,49.170


,English,Reference Igbo,Predicted Igbo
0,the lord god made a woman from the rib which h...,"mgbe ahụ , onyenwe anyị chineke ji otu ọgịrịga...",onyenwe anyị chineke mere nwanyị site n ' oghe...
1,others jailed includes ; adamu mohammed who is...,ụfọdụ ndị ọzọ atụrụ mkpọrọ bụ adamu mohammed d...,ndị ọzọ a tụrụ mkpọrọ bụ ; mma mohammed bụ afọ...
2,for who makes you different ? and what do you ...,onye gwara gị na ị pụrụ iche n ' etiti mmadụ n...,n ' ihi na onye na - eme ka unu dị iche ? gịnị...
3,"you want to buy something , go to these places","ị chọrọ izu ihe , gaa ebe ndị a","ị chọrọ ịzụ ihe , gaa n ' ebe ndị a"
4,this company has stopped work there and sacked...,ụlọọrụ a akwụsịla ọrụ n ' ebe ahụ ma chụpụ ndị...,ụlọ ọrụ a kwụsịrị ọrụ n ' ebe ahụ ma chụpụ ndị...
5,"after about one hour passed , another confiden...","mgbe ihe dị ka otu awa gafere , onye ọzọ bịakw...","mgbe ihe dị ka otu awa gafere , obi siri ike k..."
6,"he found a fresh jawbone of a donkey , put out...","mgbe ahụ samsin lere anya gburugburu , hụ ọkpụ...","ọ hụrụ ka ịnyịnya ibu ahụ dị mma , tinye aka y..."
7,but he abandoned the counsel of the old men wh...,"ma ọ nabataghị ndụmọdụ ndị okenye nyere ya , ọ...",ma ọ hapụrụ ndụmọdụ ndị ahụ niile ha nyere ya ...
8,the federal government could have simply incre...,gọọmentị etiti gaara enyekwu ikike nye ụlọ ọrụ...,gọọmentị etiti nwere ike ime ka ndị dị ugbu a ...
9,"stop attacking shoprite in nigeria , they ' re...",kwụsị ịwakpo oke ụlọahịa na mba naịjirịa . ọ b...,"kwụsị mwakpo oke egwu na naịjirịa , ha na - en..."


Metrics path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_test_metrics.csv
Preview path: /content/drive/MyDrive/CS156 Final Assignment/Transformer Outputs/transformer_all_baseline_test_preview.csv


In [17]:
sample_sentences = [
    "good morning",
    "god is good",
    "where are you going",
    "the lord is my shepherd",
]

for sentence in sample_sentences:
    prediction = decode_sequence(model_to_evaluate, sentence)
    print(f"English: {sentence}")
    print(f"Predicted Igbo: {prediction}")
    print("-" * 60)


English: good morning
Predicted Igbo: ezi ụtụtụ
------------------------------------------------------------
English: god is good
Predicted Igbo: chineke dị mma
------------------------------------------------------------
English: where are you going
Predicted Igbo: ebe ị na - aga
------------------------------------------------------------
English: the lord is my shepherd
Predicted Igbo: onyenwe anyị bụ onye ọzụzụ atụrụ m
------------------------------------------------------------
